# Stellar mass calculations and table generator

In [ ]:
import pickle
import numpy as np
import os
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
from astropy import constants as const

# Cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)


def get_median_and_uncertainties(samples):
    median = np.percentile(samples, 50)
    lower = median - np.percentile(samples, 16)
    upper = np.percentile(samples, 84) - median
    return median, lower, upper

def summarize_value(med, lo, up, fmt=".3f"):
    return f"${med:{fmt}}^{{+{up:{fmt}}}}_{{-{lo:{fmt}}}}$"

def sigma_crit(z_l, z_s):
    D_d = cosmo.angular_diameter_distance(z_l).to(u.kpc)
    D_s = cosmo.angular_diameter_distance(z_s).to(u.kpc)
    D_ds = cosmo.angular_diameter_distance_z1z2(z_l, z_s).to(u.kpc)
    sig = (const.c**2 / (4*np.pi*const.G)) * (D_s / (D_d * D_ds))
    return sig.to(u.Msun/u.kpc**2).value, D_d.value

def einstein_radius_kpc(theta_E_arcsec, Dd_kpc):
    theta_rad = np.deg2rad(theta_E_arcsec / 3600.)
    return theta_rad * Dd_kpc

def sis_sigma(theta_E_arcsec, z_l, z_s):
    theta_rad = np.deg2rad(theta_E_arcsec / 3600.)
    Ds = cosmo.angular_diameter_distance(z_s)
    Dds = cosmo.angular_diameter_distance_z1z2(z_l, z_s)
    factor = (theta_rad * Ds / (4*np.pi * Dds)).to(u.dimensionless_unscaled)
    sigma = (const.c * np.sqrt(factor)).to(u.km/u.s)
    return sigma.value

names = ['J0407-5006', 'J0806+2006', 'J1001+5027',
         'J1442+4055', 'J1515+1511', 'J1620+1203', 'J2325-5229']

z_lens_dict = {
    'J0407-5006': 0.55,
    'J0806+2006': 0.573,
    'J1001+5027': 0.415,
    'J1442+4055': 0.284,
    'J1515+1511': 0.742,
    'J1620+1203': 0.398,
    'J2325-5229': 0.40
}

z_src_dict = {
    'J0407-5006': 1.515,
    'J0806+2006': 1.54,
    'J1001+5027': 1.838,
    'J1442+4055': 2.593,
    'J1515+1511': 2.049,
    'J1620+1203': 1.158,
    'J2325-5229': 2.739
}


table_rows = []

# Loop through systems
for name in names:

    filename = f"../joint_modeling/{name}/{name}_joint.pkl"
    if not os.path.exists(filename):
        print(f"Missing {filename}")
        continue

    with open(filename, "rb") as f:
        loaded = pickle.load(f)

    chain_list = loaded["chain_list"]
    sampler_type, samples_mcmc, param_mcmc, dist_mcmc = chain_list[3]

    # Extract theta_E
    idx = param_mcmc.index("theta_E_lens0")
    theta_chain = samples_mcmc[:, idx]

    # Critical surface density (fixed)
    z_l = z_lens_dict[name]
    z_s = z_src_dict[name]
    sigcrit, Dd_kpc = sigma_crit(z_l, z_s)

    # Compute derived quantities for each sample
    R_chain = np.array([einstein_radius_kpc(t, Dd_kpc) for t in theta_chain])
    M_chain = np.pi * R_chain**2 * sigcrit
    logM_chain = np.log10(M_chain)
    sigma_chain = np.array([sis_sigma(t, z_l, z_s) for t in theta_chain])

    # Summaries
    R_med, R_lo, R_up = get_median_and_uncertainties(R_chain)
    M_med, M_lo, M_up = get_median_and_uncertainties(logM_chain)
    sig_med, sig_lo, sig_up = get_median_and_uncertainties(sigma_chain)

    # Format LaTeX
    R_tex   = summarize_value(R_med, R_lo, R_up, fmt=".3f")
    Sig_tex = f"${np.log10(sigcrit):.3e}$"        # No uncertainties
    M_tex   = summarize_value(M_med, M_lo, M_up, fmt=".3f")
    S_tex   = summarize_value(sig_med, sig_lo, sig_up, fmt=".1f")

    table_rows.append(f"{name} & {Sig_tex} & {M_tex} & {S_tex} \\\\")

# Print LaTeX table
print("\\begin{table}[ht]")
print("\\centering")
print("\\caption{Einstein radius, critical surface density, Einstein mass, and velocity dispersion for each lens system.}")
print("\\label{tab:einstein_mass_results}")
print("\\begin{tabular}{lcccc}")
print("\\hline")
print("System & $\\Sigma_{\\rm crit}$ & $\\log_{10}(M(<\\theta_E))$ & $\\sigma_{\\rm SIS}$ \\\\")
print(" & (M$_\\odot$/kpc$^2$) & (M$_\\odot$) & (km/s) \\\\")
print("\\hline")

for row in table_rows:
    print(row)

print("\\hline")
print("\\end{tabular}")
print("\\end{table}")



\begin{table}[ht]
\centering
\caption{Einstein radius, critical surface density, Einstein mass, and velocity dispersion for each lens system.}
\label{tab:einstein_mass_results}
\begin{tabular}{lcccc}
\hline
System & $\Sigma_{\rm crit}$ & $\log_{10}(M(<\theta_E))$ & $\sigma_{\rm SIS}$ \\
 & (M$_\odot$/kpc$^2$) & (M$_\odot$) & (km/s) \\
\hline
J0407-5006 & $9.373e+00$ & $11.329^{+0.008}_{-0.006}$ & $233.3^{+1.1}_{-0.8}$ \\
J0806+2006 & $9.373e+00$ & $11.164^{+0.010}_{-0.009}$ & $212.2^{+1.2}_{-1.1}$ \\
J1001+5027 & $9.337e+00$ & $11.502^{+0.006}_{-0.005}$ & $252.5^{+0.9}_{-0.8}$ \\
J1442+4055 & $9.367e+00$ & $11.175^{+0.002}_{-0.002}$ & $212.7^{+0.3}_{-0.3}$ \\
J1515+1511 & $9.344e+00$ & $11.423^{+0.004}_{-0.005}$ & $242.2^{+0.6}_{-0.7}$ \\
J1620+1203 & $9.415e+00$ & $11.667^{+0.004}_{-0.004}$ & $290.3^{+0.7}_{-0.7}$ \\
J2325-5229 & $9.304e+00$ & $11.612^{+0.001}_{-0.002}$ & $263.9^{+0.2}_{-0.2}$ \\
\hline
\end{tabular}
\end{table}
